# ResNet-50 Baseline: train -> predictions -> eval -> figures

Run this notebook in Google Colab with GPU enabled. It assumes the ResNet-50 baseline code is pushed to the selected Git branch.

## 1. Clone the project branch

Change `BRANCH` if you push the code to another branch.

In [ ]:
BRANCH = "vasily/add-visualization"
!git clone --branch {BRANCH} https://github.com/Eroouu/DL_project.git
%cd DL_project
!git status --short --branch

## 2. Install minimal dependencies for ResNet baseline

Colab already includes PyTorch/torchvision in most runtimes. We install the lightweight project dependencies needed for data/eval/plots.

In [ ]:
!pip install -q pandas numpy pillow scikit-learn matplotlib pyyaml

## 3. Provide metadata and images

For eval-only checks, `metadata/metadata_val.csv` is enough. For training ResNet-50, the RGB images referenced by `image_path` must also exist.

Option A: upload `metadata.rar` / `metadata.zip` if Erik gave it to you.

Option B: mount Google Drive and run `src.prepare_acdc` if you have `gt_trainval.zip` and `rgb_anon_trainvaltest.zip` there.

In [ ]:
# Option A: upload metadata archive from your computer.
from google.colab import files
uploaded = files.upload()
print(uploaded.keys())

In [ ]:
# Run only the command that matches your uploaded archive name.
# If you uploaded metadata.rar:
!apt-get update -qq && apt-get install -y -qq unrar
!test -f metadata.rar && unrar x -o+ metadata.rar . || true

# If you uploaded metadata.zip:
!test -f metadata.zip && unzip -o metadata.zip || true

!find metadata -maxdepth 2 -type f | sort | head -20

In [ ]:
# Option B: if you have ACDC ZIP archives in Google Drive, mount and build metadata/data in Colab.
# Uncomment and adjust paths if needed.
# from google.colab import drive
# drive.mount('/content/drive')
# !python -m src.prepare_acdc \
#   --gt-zip /content/drive/MyDrive/acdc/gt_trainval.zip \
#   --rgb-zip /content/drive/MyDrive/acdc/rgb_anon_trainvaltest.zip \
#   --data-root data/acdc \
#   --metadata-dir metadata \
#   --prefix metadata

## 4. Validate metadata and image paths

In [ ]:
!python scripts/validate_data_contract.py --metadata-dir metadata --classmap configs/classmap.json

In [ ]:
import pandas as pd
from pathlib import Path

val_df = pd.read_csv('metadata/metadata_val.csv')
display(val_df.head())
print('val rows:', len(val_df))
print('first image_path:', val_df.loc[0, 'image_path'])
print('first image exists:', Path(val_df.loc[0, 'image_path']).exists())

If `first image exists` is `False`, ResNet training cannot run yet. You need the ACDC RGB images in the paths used by `metadata_val.csv`, or regenerate metadata in this Colab runtime with `src.prepare_acdc`.

## 5. Eval pipeline check on real metadata with dummy predictions

In [ ]:
!python src/eval.py \
  --metadata metadata/metadata_val.csv \
  --dummy \
  --tune-thresholds \
  --out reports/metrics/metadata_val_dummy_eval.json

!python src/visualize.py \
  --metrics reports/metrics/metadata_val_dummy_eval.json \
  --out-dir reports/figures

## 6. Train ResNet-50 smoke baseline

This is a short run using `limit_train` / `limit_val` from `configs/resnet50_smoke.yaml`.

In [ ]:
!python scripts/train.py --config configs/resnet50_smoke.yaml --metadata-dir metadata --output-dir checkpoints --device auto

## 7. Export ResNet predictions to NPZ

In [ ]:
!python scripts/predict.py \
  --checkpoint checkpoints/resnet50_epoch01.pt \
  --metadata-dir metadata \
  --out artifacts/predictions/resnet50_val.npz \
  --device auto

## 8. Evaluate ResNet-50 baseline and build figures

In [ ]:
!python src/eval.py \
  --metadata metadata/metadata_val.csv \
  --predictions artifacts/predictions/resnet50_val.npz \
  --class-map configs/classmap.json \
  --tune-thresholds \
  --out reports/metrics/resnet50_val.json

!python src/visualize.py \
  --metrics reports/metrics/resnet50_val.json \
  --out-dir reports/figures

## 9. Optional: full ResNet-50 run

Run this only after the smoke baseline works.

In [ ]:
# !python scripts/train.py --config configs/resnet50_full.yaml --metadata-dir metadata --output-dir checkpoints --device auto

## 10. Package outputs for report

In [ ]:
!zip -r resnet50_outputs.zip reports/metrics reports/figures artifacts/predictions
files.download('resnet50_outputs.zip')